In [1]:
!pip install gradio mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 10.0 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [2]:
import gradio as gr
import cv2
import numpy as np
import mediapipe as mp
import json
import gdown
import os
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [3]:
model_path = '/content/fix_bisindo_finetuned.keras'
class_path = '/content/fix_class_indices.json'

gdown.download(id='1HgMAC_H2-VHOVOke0tXzUSWLI4A7GtnJ', output=model_path, quiet=False, fuzzy=True)
gdown.download(id='1ZHCWoWiew5u6DoZLYFApmdRaaNVHbl7L', output=class_path, quiet=False, fuzzy=True)

print("Model size :", os.path.getsize(model_path), "bytes")
print("JSON size  :", os.path.getsize(class_path), "bytes")

Downloading...
From (original): https://drive.google.com/uc?id=1HgMAC_H2-VHOVOke0tXzUSWLI4A7GtnJ
From (redirected): https://drive.google.com/uc?id=1HgMAC_H2-VHOVOke0tXzUSWLI4A7GtnJ&confirm=t&uuid=6561c297-c383-473d-9c25-ac5e3c42a3af
To: /content/fix_bisindo_finetuned.keras
100%|██████████| 223M/223M [00:04<00:00, 50.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1ZHCWoWiew5u6DoZLYFApmdRaaNVHbl7L
To: /content/fix_class_indices.json
100%|██████████| 224/224 [00:00<00:00, 1.10MB/s]

Model size : 223381605 bytes
JSON size  : 224 bytes


In [4]:
!wget -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2)
detector = vision.HandLandmarker.create_from_options(options)

In [5]:
model = load_model(model_path)
model.summary()

Model: "ResNet50_BISINDO"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 55,675,344 (212.38 MB)

 Trainable params: 15,512,602 (59.18 MB)

 Non-trainable params: 9,137,536 (34.86 MB)

 Optimizer params: 31,025,206 (118.35 MB)

In [6]:
with open(class_path) as f:
    class_indices = json.load(f)

class_names = {v: k for k, v in class_indices.items()}
print('Jumlah Kelas:', len(class_names))

Jumlah Kelas: 26


In [7]:
def detect(frame):
    if frame is None:
        return (
            '<div class="prediction-text">-</div>',
            '<div class="conf-text">Confidence: -</div>')
    try:
        # =========================
        # MEDIAPIPE
        # =========================
        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=frame)

        detection_result = detector.detect(mp_image)

        if not detection_result.hand_landmarks:
            return (
                '<div class="prediction-text">-</div>',
                '<div class="conf-text">Tangan tidak terdeteksi</div>'
            )

        hand_landmarks = detection_result.hand_landmarks[0]

        h, w, _ = frame.shape

        x_coords = [int(lm.x * w) for lm in hand_landmarks]
        y_coords = [int(lm.y * h) for lm in hand_landmarks]

        x_min, x_max = min(x_coords), max(x_coords)
        y_min, y_max = min(y_coords), max(y_coords)

        padding = 20

        x_min = max(0, x_min - padding)
        y_min = max(0, y_min - padding)
        x_max = min(w, x_max + padding)
        y_max = min(h, y_max + padding)

        hand_crop = frame[y_min:y_max, x_min:x_max]

        if hand_crop.size == 0:
            return (
                '<div class="prediction-text">-</div>',
                '<div class="conf-text">Crop gagal</div>'
            )
        # =========================
        # PREPROCESS RESNET50
        # =========================
        hand_crop = cv2.resize(hand_crop, (224,224))

        hand_crop = preprocess_input(hand_crop.astype(np.float32))
        hand_crop = np.expand_dims(hand_crop, axis=0)

        pred = model.predict(hand_crop, verbose=0)

        class_id = int(np.argmax(pred))
        confidence = float(np.max(pred)) * 100

        label_text = class_names[class_id]

        if confidence < 50:
            label_text = ""

        return (
            f'<div class="prediction-text">{label_text}</div>',
            f'<div class="conf-text">Confidence: {confidence:.2f}%</div>'
        )

    except Exception as e:
        return (
            '<div class="prediction-text">Error</div>',
            f'<div class="conf-text">{str(e)}</div>'
        )

In [8]:
custom_css = """
.navbar {
    background: white;
    padding: 15px 30px;
    box-shadow: 0 2px 6px rgba(0,0,0,0.05);
    display: flex;
    justify-content: space-between;
    align-items: center;
}

.result-card {
    background: white;
    border-radius: 20px;
    padding: 40px;
    box-shadow: 0 10px 25px rgba(0,0,0,0.08);
    text-align: center;
}

.prediction-text {
    font-size: 70px;
    font-weight: bold;
    color: #3b5bdb;
    margin: 30px 0;
}

.conf-text {
    font-size: 16px;
    color: gray;
}
"""

In [9]:
with gr.Blocks(css=custom_css) as demo:

    gr.Markdown("## Sistem Deteksi BISINDO")

    with gr.Row():

        camera = gr.Image(
            sources="webcam",
            streaming=True,
            type="numpy")

        with gr.Column(elem_classes="result-card"):
            gr.Markdown("### Hasil Prediksi")
            label_output = gr.HTML('<div class="prediction-text">-</div>')
            conf_output = gr.HTML('<div class="conf-text">Confidence: -</div>')

    camera.stream(
        detect,
        inputs=camera,
        outputs=[label_output, conf_output])

demo.launch()

/tmp/ipykernel_10644/858017898.py:1: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1a92c2449a85c36723.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
